### Launch the ROS demo

In [ ]:
%%bash --bg
trap 'kill $(jobs -p)' EXIT
xdg-terminal-exec -- bash -l -c "source /opt/ros/lyrical/setup.bash && ros2 launch demo.launch.xml"

Create the controller and run it periodically in a thread

In [ ]:
# Import rclpy and create a ROS2 node
import rclpy
from rclpy.node import Node
from threading import Thread
from controller import Controller
from interactive_markers.interactive_marker_server import InteractiveMarkerServer
from markers import addMarker, processFeedback, iPoseMarker

rclpy.init()
node = Node("ns_demo")

c = Controller(node)
ims = InteractiveMarkerServer(node, "controller")
addMarker(ims, iPoseMarker(c.T), processFeedback(c.setTarget))
ims.applyChanges()

In [ ]:
class Worker(Thread):
    def __init__(self, *args, **kwargs):
        super(Worker, self).__init__(*args, **kwargs)
        self.stop = False

    def run(self):
        executor = rclpy.get_global_executor()
        executor.add_node(node)
        while executor.context.ok() and not self.stop:
            c.pose_control(c.targets["pose"])
            executor.spin_once(timeout_sec=0.02)

try:  # stop previous thread if existing
    t.stop = True
except NameError:
    pass

t = Worker()
t.start()

Create a list of slider widgets, one for each joint, to chose the default pose

In [ ]:
from ipywidgets import FloatSlider, Layout, Button, Box
joint_widgets = [FloatSlider(min = j.min, max = j.max, step = (j.max-j.min) / 100, description = j.name) \
                 for j in c.robot.active_joints]

React to slider (value) changes by adapting the default joint pose

In [ ]:
def on_sent(event):
    widget = event.owner
    c.preferred_joints[c.joint_msg.name.index(widget.description)] = widget.value

for widget in joint_widgets:
    widget.observe(on_sent, 'value')

Collect all widgets (sliders and buttons) in a form and display them

In [ ]:
form = Box(joint_widgets, layout=Layout(
    display='flex',
    flex_flow='column',
    border='solid 2px',
    align_items='stretch',
    width='100%'
))
display(form)